In [26]:
import pandas as pd
import numpy as np

# Load datasets
books = pd.read_csv("books.csv", dtype=str, encoding="Windows-1252")  # Ensure text data is read correctly
ratings = pd.read_csv("ratings.csv", dtype=str)
users = pd.read_csv("users.csv", dtype=str)

# Display the first few rows
print(books.head())
print(ratings.head())
print(users.head())


         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                   Publisher  \
0    Mark P. O. Morford                2002     Oxford University Press   
1  Richard Bruce Wright                2001       HarperFlamingo Canada   
2          Carlo D'Este                1991             HarperPerennial   
3      Gina Bari Kolata                1999        Farrar Straus Giroux   
4       E. J. W. Barber                1999  W. W. Norton &amp; Company   

                                         Image-URL-S  \
0  http://images.amazon.com/images/P/0195153448.0...   
1  http://images.amazon.com/

In [27]:
# Drop image URL columns
books.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"], inplace=True)

# Convert Year-Of-Publication to numeric, setting invalid values to NaN
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")

# Replace invalid years (e.g., future years) with the median year
valid_years = books[(books["Year-Of-Publication"] >= 1000) & (books["Year-Of-Publication"] <= 2025)]
median_year = valid_years["Year-Of-Publication"].median()
books["Year-Of-Publication"] = books["Year-Of-Publication"].fillna(median_year)
books.loc[books["Year-Of-Publication"] > 2025, "Year-Of-Publication"] = median_year  # Fix future years

# Handle missing values (fill missing authors/publishers with "Unknown")
books.fillna({"Book-Author": "Unknown", "Publisher": "Unknown"}, inplace=True)

print(books.info())  # Check cleaned data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ISBN                 271360 non-null  object 
 1   Book-Title           271360 non-null  object 
 2   Book-Author          271360 non-null  object 
 3   Year-Of-Publication  271360 non-null  float64
 4   Publisher            271360 non-null  object 
dtypes: float64(1), object(4)
memory usage: 10.4+ MB
None


In [28]:
# Convert Book-Rating to integer
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce").astype(int)

# Remove implicit ratings (0 ratings) if necessary
ratings = ratings[ratings["Book-Rating"] > 0]

# Convert User-ID and ISBN to categorical (for modeling)
ratings["User-ID"] = ratings["User-ID"].astype("category")
ratings["ISBN"] = ratings["ISBN"].astype("category")

print(ratings.info())  # Check cleaned ratings data

<class 'pandas.core.frame.DataFrame'>
Index: 433671 entries, 1 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype   
---  ------       --------------   -----   
 0   User-ID      433671 non-null  category
 1   ISBN         433671 non-null  category
 2   Book-Rating  433671 non-null  int32   
dtypes: category(2), int32(1)
memory usage: 16.3 MB
None


In [29]:
# Replace "n/a" and empty values with NaN
users.replace({"n/a": np.nan, "": np.nan}, inplace=True)

# Split 'Location' into City, State, Country safely
users[["City", "State", "Country"]] = users["Location"].str.split(", ", expand=True, n=2)

# Drop the old 'Location' column
users.drop(columns=["Location"], inplace=True)

# If country is missing but state exists, assume it's invalid
users.loc[users["Country"].isna() & users["State"].notna(), "Country"] = np.nan

# If state is missing but country exists, assume it's valid
users.loc[users["State"].isna() & users["Country"].notna(), "State"] = "Unknown"

# Handle cases like "ferrol / spain, alabama, spain"
users["City"] = users["City"].str.replace(r" / .*", "", regex=True)  # Keep only first city name

# Display cleaned data
print(users.head())

# Convert Age to numeric, fill missing with median
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users["Age"] = users["Age"].fillna(users["Age"].median())

print(users.info())  # Check cleaned users data

  User-ID   Age         City            State         Country
0       1   NaN          nyc         new york             usa
1       2  18.0     stockton       california             usa
2       3   NaN       moscow  yukon territory          russia
3       4  17.0        porto         v.n.gaia        portugal
4       5   NaN  farnborough            hants  united kingdom
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   User-ID  278858 non-null  object 
 1   Age      278858 non-null  float64
 2   City     278858 non-null  object 
 3   State    278857 non-null  object 
 4   Country  274281 non-null  object 
dtypes: float64(1), object(4)
memory usage: 10.6+ MB
None


In [30]:
# Function to detect mojibake

def has_mojibake(text):
    if isinstance(text, str):
        return "Ã" in text or "Â" in text or "�" in text  # Common mojibake artifacts
    return False

# Drop rows where Title or Publisher is corrupted
books_cleaned = books[~books["Book-Title"].apply(has_mojibake)]
books_cleaned = books_cleaned[~books_cleaned["Publisher"].apply(has_mojibake)]

# Save cleaned dataset
books_cleaned.to_csv("cleaned_books.csv", index=False, encoding="utf-8")

print(f"Original dataset: {len(books)} rows")
print(f"Cleaned dataset: {len(books_cleaned)} rows")
print(f"Dropped rows: {len(books) - len(books_cleaned)}")

Original dataset: 271360 rows
Cleaned dataset: 263732 rows
Dropped rows: 7628


In [31]:
# Save cleaned datasets to CSV files
users.to_csv("cleaned_users.csv", index=False)
ratings.to_csv("cleaned_ratings.csv", index=False)